using the notebook to try our LLM-EWF

In [70]:
import functools
import re
import pickle
import os
import openai

from collections import deque
from typing import Literal, Deque
from typing import TypedDict

from langchain import hub
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import BaseMessage, AIMessage, SystemMessage
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from pydantic.v1 import BaseModel
from copy import deepcopy

'''customized tool import'''
# emergency_department
from tools.emergency_management_agent import *

# meteorological_bureau
from tools.meteorology_agent import *
# hydrological_department
from tools.hydrology_agent import *
# natural_resource_department
from tools.natural_resource_agent import *

# rural_department
from tools.agriculture_agent import *
# environment
from tools.situation_awareness_agent.environmentMonitor_tool import MonitorStartTool
from tools.situation_awareness_agent.retrieveLog_tool import LogRetrieveTool
from tools.situation_awareness_agent.storeSubtask_tool import StoreSubtaskTool

# other functional agent
# ...

# root of file
from base_config.root_base import *
from base_config.prompt_config import *


In [74]:
# Define a helper function that we will use to create the nodes in the graph - it takes care of converting the agent
# response to a human message. This is important because that is how we will add it the global state of the graph
def agent_node(state, agent, name):
    print()
    result = agent.invoke(state)
    filtered_messages = [
        HumanMessage(
            content=msg.content)
        if isinstance(msg, HumanMessage)
        else AIMessage(
            content=msg.content, name=msg.name if isinstance(msg.name, str) and re.match(r'^[a-zA-Z0-9_-]+$', msg.name)
            else name)
        for msg in result["messages"] if isinstance(msg, (HumanMessage, AIMessage))
    ]
    print('###')
    print(name)
    return {"messages": filtered_messages if filtered_messages else []}

# The agent state is the input to each node in the graph
class AgentState(TypedDict):
    # The annotation tells the graph that new messages will always
    # be added to the current states
    # messages: Annotated[Sequence[BaseMessage], operator.add]

    # Store the last few messages
    messages: Deque[BaseMessage]
    # The 'next' field indicates where to route to next
    next: str
    # The 'subtask' field indicates what to do next
    subtask: str


'''gpt-4o as the base model'''
openai.api_key = os.environ['OPENAI_API_KEY']
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# define the functional department
'''meteorological_bureau define'''
meteorological_alert_tool = MeteorologicalAlertTool()
meteorological_forecast_tool = MeteorologicalForecastTool()
meteorological_emergency_response_tool = MeteorologicalResponseTool()
meteorological_retrieve_tool = MeteorologicalRetrieveTool()
meteorological_visualize_tool = MeteorologicalVisualizeTool()

tools_mete = [meteorological_alert_tool, meteorological_forecast_tool, meteorological_emergency_response_tool,
              meteorological_retrieve_tool, meteorological_visualize_tool]
meteorological_bureau = create_react_agent(llm, tools_mete, state_modifier=prompt_meteorology)
meteorological_bureau_node = functools.partial(agent_node, agent=meteorological_bureau, name="Meteorological_Bureau")

'''emergency_department define'''
ap_forecast_tool = APForecastTool()
population_evacuation_tool = AffectedPopulationRelocationTool()
material_dispatch_tool = MaterialDispatchTool()
population_alert_tool = PopulationAlertTool()
establish_emergency_command_center_tool = EstablishEmergencyCommandCenterTool()
emergency_response_assess_tool = EmergencyResponseAssessTool()
initiate_emergency_response_tool = ActivateEmergencyResponseTool()
terminate_emergency_response_tool = TerminateEmergencyResponseTool()
draft_disaster_report_tool = DraftDisasterReportTool()
emergency_retrieve_tool = EmergencyCenterRetrieveTool()
emergency_visualize_tool = EmergencyCenterVisualizationTool()

tools_emer = [ap_forecast_tool, population_evacuation_tool, material_dispatch_tool, population_alert_tool,
              establish_emergency_command_center_tool, emergency_response_assess_tool, initiate_emergency_response_tool,
              terminate_emergency_response_tool, draft_disaster_report_tool, emergency_retrieve_tool, emergency_visualize_tool]

emergency_department = create_react_agent(llm, tools_emer, state_modifier=prompt_emergency_management)
emergency_department_node = functools.partial(agent_node, agent=emergency_department, name="Emergency_Management_Department")

'''hydrological_department define'''
hydrological_forecast_tool = HydrologicalForecastTool()
hydraulic_forecast_tool = HydraulicForecastTool()
hydrological_alert_tool = HydrologicalAlertTool()
hydrological_response_tool = HydrologicalResponseTool()
hydraulic_repair_tool = HydraulicRepairTool()
hydrological_damage_survey_tool = HydrologicalDamageSurveyTool()
hydrological_disaster_report_tool = HydrologicalDisasterReportTool()
hydrological_retrieve_tool = HydrologicalRetrieveTool()
hydrological_visualize_tool = HydrologicalVisualizeTool()
tools_hydro = [hydrological_forecast_tool, hydraulic_forecast_tool, hydrological_alert_tool, hydrological_response_tool,
               hydraulic_repair_tool, hydrological_damage_survey_tool, hydrological_disaster_report_tool,
               hydrological_retrieve_tool, hydrological_visualize_tool]

hydrological_department = create_react_agent(llm, tools_hydro, state_modifier=prompt_hydrology)
hydrological_department_node = functools.partial(agent_node, agent=hydrological_department, name="Hydrological_Department")

'''natural_resource_department define'''
natural_resource_forecast_tool = NaturalResourceForecastTool()
natural_resource_warning_tool = NaturalResourceAlertTool()
natural_resource_response_tool = NaturalResourceResponseTool()
natural_resource_retrieve_tool = NaturalResourceRetrieveTool()
natural_resource_visualize_tool = NaturalResourceVisualizeTool()
tools_nat = [natural_resource_forecast_tool, natural_resource_warning_tool, natural_resource_response_tool, natural_resource_retrieve_tool,
             natural_resource_visualize_tool]

natural_resource_department = create_react_agent(llm, tools_nat, state_modifier=prompt_natural_resource)
natural_resource_department_node = functools.partial(agent_node, agent=natural_resource_department, name="Natural_Resource_Department")


'''rural_department'''
agricultural_forecast_tool = AgriculturalForecastTool()
agricultural_warning_tool = AgriculturalAlertTool()
agricultural_recovery_tool = AgriculturalRecoveryTool()
agricultural_survey_tool = AgriculturalDamageSurveyTool()
agricultural_report_tool = AgriculturalDisasterReportTool()
agricultural_retrieve_tool = AgriculturalDisasterRetrieveTool()
agricultural_visualize_tool = AgriculturalDamageVisualizeTool()
tools_rural = [agricultural_forecast_tool, agricultural_warning_tool, agricultural_survey_tool, agricultural_recovery_tool,
               agricultural_report_tool, agricultural_retrieve_tool, agricultural_visualize_tool]

rural_department = create_react_agent(llm, tools_rural, state_modifier=prompt_agericulture)
rural_department_node = functools.partial(agent_node, agent=rural_department, name="Rural_Department")

''' Situation Awareness Agent define'''
environment_monitor_start_tool = MonitorStartTool()
log_retrieve_tool = LogRetrieveTool()
store_subtask_tool = StoreSubtaskTool()
tools_env = [environment_monitor_start_tool, log_retrieve_tool, store_subtask_tool]

environment = create_react_agent(llm, tools_env, state_modifier=prompt_saa)
environment_node = functools.partial(agent_node, agent=environment, name="Environment")

'''Executive Agent define'''
members = ["Meteorological_Bureau", "Emergency_Management_Department", "Hydrological_Department",
           "Natural_Resource_Department", "Environment", "Rural_Department"]
options = ["FINISH"] + members
prompt_supervisor = ChatPromptTemplate.from_messages(
    [
        ("system", prompt_ea),
        MessagesPlaceholder(variable_name="messages"),
        (
            "system",
            "reminder to respond in a JSON blob no matter what",
        ),
    ]
).partial(options=str(options), members=", ".join(members))

class RouteResponse(BaseModel):
    next: Literal[tuple(options)]
    subtask: str = ''

'''planner define'''
prompt_planner = hub.pull('ideal/planner')


def calculate_subtask_queue_num(subtask_queue: dict):
    num_log = 0
    for s_q in subtask_queue:
        if s_q['status'] == 'pending':
            num_log += 1
    return num_log

def analyse_subtask_list(state):
    with open(SYSTEM_FILE_PATH, 'rb') as f:
        system_memory = pickle.load(f)
    content = deepcopy(system_memory)
    subtask = system_memory['subtask_queue']
    # print(f'origin_subtask is {subtask}\n')
    if calculate_subtask_queue_num(subtask) == 0:
        content['subtask_queue'] = deque()
        if system_memory['input_task'] != 'start':
            content['subtask_queue'].append('FINISH')
        state["messages"].append(SystemMessage(str(content)))
        # print(f"supervisor input is {state}")
    else:
        for st_ in subtask:
            if st_['status'] == 'pending':
                content['subtask_queue'] = st_
                state["messages"].append(SystemMessage(str(content)))
                print(f"supervisor input is {state}")
                st_['status'] = 'completed'
                break
    # print(f'subtask is {subtask}\n')
    # print(f'content is {content}\n')
    with open(SYSTEM_FILE_PATH, 'wb') as f:
        pickle.dump(system_memory, f)
    return state

def decompose_task():
    with open(SYSTEM_FILE_PATH, 'rb') as f:
        system_memory = pickle.load(f)
    if system_memory['subtask_queue'] == deque([]):
        input_t = system_memory['input_task']
        if input_t != 'start':
            planner = prompt_planner | llm
            subtask_l = planner.invoke(input_t).content.split('\n')
            store_subtask_tool._run(subtask_l)
    else:
        input_t = system_memory['input_task']

def chief_commander_agent(state):
    chief_commander_chain = (
        prompt_supervisor
        | llm.with_structured_output(RouteResponse)
    )
    decompose_task()
    state = analyse_subtask_list(state)
    res = chief_commander_chain.invoke(state)
    return res

In [75]:
# define state graph
workflow = StateGraph(AgentState)
workflow.add_node("Meteorological_Bureau", meteorological_bureau_node)
workflow.add_node("Emergency_Management_Department", emergency_department_node)
workflow.add_node("Hydrological_Department", hydrological_department_node)
workflow.add_node("Natural_Resource_Department", natural_resource_department_node)
workflow.add_node("supervisor", chief_commander_agent)
workflow.add_node("Environment", environment_node)
workflow.add_node("Rural_Department", rural_department_node)

for member in members:
    workflow.add_edge(member, "supervisor")

conditional_map = {k: k for k in members}
conditional_map["FINISH"] = END
workflow.add_conditional_edges('supervisor', lambda x: x['next'], conditional_map)
# Finally, add entrypoint
workflow.add_edge(START, "supervisor")

graph = workflow.compile()

# try:
#     graph.get_graph(xray=True).draw_mermaid_png(output_file_path='./graph.png')
# except Exception as e:
#     # This requires some extra dependencies and is optional
#     print(f'error: {e}')
#     pass

def initial_system_file(input_task: str = 'start'):
    subtask_queue: deque[dict] = deque([])
    environment_observation_program: str = 'START'
    emergency_command_center_status: str = 'establish'
    emergency_response_status: str = 'START'
    emergency_response_level: int = 0
    monitor_frequency: str = '24h'
    current_time: str = '2019080920'
    # time4forecast: str = '2019080920'
    # time4alert: str = '2019081020'
    init_dict = {
        'input_task': input_task,
        'subtask_queue': subtask_queue,
        'environment_observation_program': environment_observation_program,
        'emergency_command_center_status': emergency_command_center_status,
        'emergency_response_status': emergency_response_status,
        'emergency_response_level': emergency_response_level,
        'monitor_frequency': monitor_frequency,
        'current_time': current_time,
        # 'time4forecast': time4forecast,
        # 'time4alert': time4alert,
    }
    with open(SYSTEM_FILE_PATH, 'wb') as f:
        pickle.dump(init_dict, f)

test  automated early waening

In [ ]:
thread = {"configurable": {"thread_id": "3"}}
initial_system_file()
for s in graph.stream(
    {"messages": [HumanMessage(content="start")]},
    {"recursion_limit": 100}, stream_mode="values",
):
    if "__end__" not in s:
        print(s)
        print("----")

{'messages': [HumanMessage(content='start', additional_kwargs={}, response_metadata={})]}
----
{'messages': [HumanMessage(content='start', additional_kwargs={}, response_metadata={}), SystemMessage(content="{'input_task': 'start', 'subtask_queue': deque([]), 'environment_observation_program': 'START', 'emergency_command_center_status': 'establish', 'emergency_response_status': 'START', 'emergency_response_level': 0, 'monitor_frequency': '24h', 'current_time': '2019080920'}", additional_kwargs={}, response_metadata={})], 'next': 'Environment', 'subtask': 'Start Environment Monitoring'}
----

retrieve the log file
No file update records found in the log file.
{'input_task': 'start', 'subtask_queue': deque([]), 'environment_observation_program': 'START', 'emergency_command_center_status': 'establish', 'emergency_response_status': 'START', 'emergency_response_level': 0, 'monitor_frequency': '24h', 'current_time': '2019080920'}
No file update records found in the log file.
{'input_task': 's

In [76]:
thread = {"configurable": {"thread_id": "3"}}
initial_system_file()
for s in graph.stream(
    {"messages": [HumanMessage(content="start")]},
    {"recursion_limit": 100}, stream_mode="values",
):
    if "__end__" not in s:
        print(s)
        print("----")

{'messages': [HumanMessage(content='start', additional_kwargs={}, response_metadata={})]}
----
{'messages': [HumanMessage(content='start', additional_kwargs={}, response_metadata={}), SystemMessage(content="{'input_task': 'start', 'subtask_queue': deque([]), 'environment_observation_program': 'START', 'emergency_command_center_status': 'establish', 'emergency_response_status': 'START', 'emergency_response_level': 0, 'monitor_frequency': '24h', 'current_time': '2019080920'}", additional_kwargs={}, response_metadata={})], 'next': 'Environment', 'subtask': 'Start Environment Monitoring'}
----

retrieve the log file
###
Environment
{'messages': [HumanMessage(content='start', additional_kwargs={}, response_metadata={}), AIMessage(content='The environment observation program is already activated. I will now proceed to check for any updates in the environment folder.', additional_kwargs={}, response_metadata={}, name='Environment'), AIMessage(content='It seems that there are no available logs

KeyboardInterrupt: 

test impact based warning to agencies support

In [7]:
thread = {"configurable": {"thread_id": "3"}}

# input_task = 'First, calculate the rainfall across the p0rovince for the next 24 hours, then use it to estimate the affected population over the next 24 hours, and visualize the distribution of the affected population.'
input_task = """plot the rainfall across the whole province for the past 24 hours."""
initial_system_file(input_task)
for s in graph.stream(
        {"messages": [
            HumanMessage(content=input_task)]},
        {"recursion_limit": 100}, stream_mode="values",
):
    if "__end__" not in s:
        print(s)
        print("----")

{'messages': [HumanMessage(content='plot the rainfall across the whole province for the past 24 hours.', additional_kwargs={}, response_metadata={})]}
----
store the subtask: ['1. Retrieve the past 24-hour rainfall monitoring data for the whole province.', '2. Visualize the rainfall data across the whole province for the past 24 hours.']
{'input_task': 'plot the rainfall across the whole province for the past 24 hours.', 'subtask_queue': deque([{'subtask': '1. Retrieve the past 24-hour rainfall monitoring data for the whole province.', 'status': 'pending'}, {'subtask': '2. Visualize the rainfall data across the whole province for the past 24 hours.', 'status': 'pending'}]), 'environment_observation_program': 'START', 'emergency_command_center_status': 'establish', 'emergency_response_status': 'START', 'emergency_response_level': 0, 'monitor_frequency': '24h', 'current_time': '2019080920'}
supervisor input is {'messages': [HumanMessage(content='plot the rainfall across the whole provinc